# Instance Impact Framework: Full-Pipeline Case Study Notebook

## Objective
This notebook executes and presents the **entire packaged pipeline** from raw pre/post input images to instance-level impact outputs.
You should expect per-building outputs from each stage: Stage 1 instance polygons with segmentation confidence, Stage 2a exposure signals (predicted population and building type), and Stage 2b damage predictions with calibrated confidence and uncertainty metrics.
The notebook culminates in a merged instance table, uncertainty-ranked cases, and polygon overlays that make each prediction interpretable in its visual context.

## Pipeline
`Stage1 -> shared subimages -> Stage2a -> Stage2b -> presentation -> overlays`

![Instance Impact Conceptualization](II-Conceptualization.png)

## Assumptions
- Environment is prepared from package dependencies.
- Input pre/post images are valid and correspond to the same tile.
- The notebook is run top-to-bottom without skipping cells.

## Scope
- Inference + presentation only (no training).
- Structured for server-kernel execution with reproducible paths and commands.

## 0.5) GeoAI Kernel Bootstrap (One-Time User-Site Setup)

Run this once **before** the pipeline cells when you are on the managed `geoai` kernel and need Stage 1 (`samgeo` + `Sam3Model`) to work.

### Why this exists
- The base kernel in `/cvmfs/.../geoai` is read-only.
- We encountered a concrete package collision: user-site `numpy 2.x` shadowed base `numpy 1.26.4`, which broke imports with `libscipy_openblas64_-ff84a88b.so` errors and cascaded into `pandas/scipy/torch/rasterio` failures.
- We install a compatible, pinned override set into writable user site `/home/jovyan/.local/geoai` to keep the runtime stable.
- This notebook expects those user-site packages to be present.

### Terminal commands (exact sequence used in this recovery)

#### Optional: purge/reset user-site profile (if environment gets corrupted)
```bash
# backup current profile first (safe rollback)
mv /home/jovyan/.local/geoai /home/jovyan/.local/geoai_backup_$(date +%Y%m%d_%H%M%S)
mkdir -p /home/jovyan/.local/geoai

# then run the bootstrap commands above from step 1
```

```bash
# 1) Target the managed geoai interpreter and user profile
export PYTHONUSERBASE=/home/jovyan/.local/geoai
GEOAI_PY=/cvmfs/iguide.purdue.edu/software/conda/geoai/bin/python

# 2) Upgrade packaging tools in user-site
$GEOAI_PY -m pip install --user --upgrade pip setuptools wheel

# 3) Keep numeric stack stable (avoid numpy 2.x ABI churn)
$GEOAI_PY -m pip install --user "numpy==1.26.4" "opencv-python-headless==4.10.0.84"

# 4) Install base runtime pieces (no dependency backtracking)
$GEOAI_PY -m pip install --user --no-deps \
  timm==1.0.25 \
  segment-anything \
  segment-geospatial \
  geoai-py \
  geopandas

# 5) Upgrade torch stack in user-site (CPU wheels in this platform run)
$GEOAI_PY -m pip install --user --index-url https://download.pytorch.org/whl/cpu \
  "torch==2.4.1" "torchvision==0.19.1" "torchaudio==2.4.1"

# 6) Install transformers from source branch (Sam3Model symbol required)
$GEOAI_PY -m pip install --user --upgrade --no-deps \
  "git+https://github.com/huggingface/transformers.git"

# 7) Patch missing runtime dependencies observed during import tests
$GEOAI_PY -m pip install --user --upgrade httpx
$GEOAI_PY -m pip install --user --upgrade "huggingface_hub>=0.30"
$GEOAI_PY -m pip install --user --upgrade "regex>=2025.10.22"
$GEOAI_PY -m pip install --user --upgrade tokenizers safetensors
```

### Validation (run in notebook)
```python
import site, numpy, torch, transformers
from transformers import Sam3Model
from samgeo import SamGeo3

print("user_site:", site.getusersitepackages())
print("numpy:", numpy.__version__, numpy.__file__)
print("torch:", torch.__version__, torch.__file__)
print("transformers:", transformers.__version__, transformers.__file__)
print("Sam3Model + SamGeo3 import: OK")
```

### Required after installation
1. **Restart the kernel** once installs finish.
2. Re-run notebook from the top.

### Notes
- Warnings like `.../home/jovyan/.local/geoai/bin is not on PATH` are expected and safe.
- Do **not** try to uninstall packages from `/cvmfs/...`; that filesystem is read-only.
- If Stage 1 OOMs, reduce Stage 1 batch size (`--batch-size 1` is the safest default).

### Core Imports

Load foundational Python modules for path handling, shell execution, and file operations.

In [ ]:
from pathlib import Path
import os
import json
import shutil
import subprocess
import shlex
import sys

print("Base imports ready.")

### HF Token Setup (manual input required)

Configure optional secure `HF_TOKEN` input required for gated Stage1 model access.

In [ ]:
# --- Token setup for Stage1 SAM3 access (public-safe) ---
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN", "").strip():
    token = getpass("Enter HF_TOKEN (input hidden, leave blank to skip): ").strip()
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN set for current kernel session.")
    else:
        print("HF_TOKEN not provided. Stage1 may fail if SAM3 model access is required.")
else:
    print("HF_TOKEN present in environment.")

### Case Configuration

Define package root, input image pair, run ID, and runtime configuration for this study.

In [ ]:
# --- 1) Study configuration: full pipeline run ---

# Auto-locate package root (supports opening notebook from parent directory).
PACKAGE_ROOT = Path.cwd()
if not (PACKAGE_ROOT / "run_pipeline.sh").exists() and (PACKAGE_ROOT / "II_package" / "run_pipeline.sh").exists():
    PACKAGE_ROOT = PACKAGE_ROOT / "II_package"

# Input image pair (customize these two paths).
PRE_IMAGE = PACKAGE_ROOT / "example_image_pair" / "nepal-flooding_00000408_pre_disaster.png"
POST_IMAGE = PACKAGE_ROOT / "example_image_pair" / "nepal-flooding_00000408_post_disaster.png"

# Run identifier for outputs.
RUN_ID = "nb_full_pipeline_nepal_flooding_00000408"

# Runtime settings (aligned with package defaults).
PYTHON_BIN = sys.executable
CUDA_VISIBLE_DEVICES = "0"
VERBOSE_RUN = False

print("PACKAGE_ROOT:", PACKAGE_ROOT)
print("PRE_IMAGE:", PRE_IMAGE)
print("POST_IMAGE:", POST_IMAGE)
print("RUN_ID:", RUN_ID)
print("PYTHON_BIN:", PYTHON_BIN)

### Run Directory Reset

Create a fresh output run directory to ensure reproducible notebook execution.

In [ ]:
# --- 2) Prepare a clean run directory ---
# Environment is expected to be prepared in the active kernel user-site.
# This cell keeps runtime setup deterministic by resetting the run folder.

RUN_ROOT = PACKAGE_ROOT / "outputs" / "driver_runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUN_ROOT / RUN_ID
if RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
print("RUN_DIR (fresh):", RUN_DIR)

### Scientific Imports and Device Setup

Import scientific libraries and resolve CPU/GPU device targets for all stages.

In [ ]:
# --- 2b) Scientific/runtime imports + device selection ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

STAGE1_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
STAGE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("numpy:", np.__version__)
print("numpy path:", np.__file__)
print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available(), "count:", torch.cuda.device_count())
print("STAGE1_DEVICE:", STAGE1_DEVICE)
print("STAGE_DEVICE:", STAGE_DEVICE)
print("Scientific imports ready.")

### Stage 1 Execution

Prepare input links and run SAM3-based building instance extraction on the pre-disaster image.

In [ ]:
# --- 3a) Stage 1: execute ---

env = dict(os.environ)
env["PYTHON_BIN"] = PYTHON_BIN
env["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES

def run_step(cmd, title):
    print(f"\n===== {title} =====")
    print("[cmd]", " ".join(shlex.quote(str(x)) for x in cmd))
    subprocess.run([str(x) for x in cmd], check=True, cwd=str(PACKAGE_ROOT), env=env)

tile_id = PRE_IMAGE.stem
if tile_id.endswith("_pre_disaster"):
    tile_id = tile_id[: -len("_pre_disaster")]

pair_dir = RUN_DIR / "pair_inputs"
pair_dir.mkdir(parents=True, exist_ok=True)
pre_link = pair_dir / f"{tile_id}_pre_disaster.png"
post_link = pair_dir / f"{tile_id}_post_disaster.png"
for dst in [pre_link, post_link]:
    if dst.exists() or dst.is_symlink():
        dst.unlink()
pre_link.symlink_to(PRE_IMAGE.resolve())
post_link.symlink_to(POST_IMAGE.resolve())

stage1_out = RUN_DIR / "stage1"
shared_out = RUN_DIR / "shared_instances_r48"
stage2a_input_csv = RUN_DIR / "stage2a_infer_input.csv"
stage2a_out_csv = RUN_DIR / "stage2a_predictions.csv"
stage2b_out_jsonl = RUN_DIR / "stage2b_ensemble.jsonl"
presented_csv = RUN_DIR / "instance_results_presented.csv"
uncertain_csv = RUN_DIR / "instance_results_top_uncertain.csv"
vis_dir = RUN_DIR / "vis_instance_level"

run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "stage1/SAM3_Final_20260226/scripts/run_sam3_building_infer.py",
        "--input", pair_dir,
        "--output", stage1_out,
        "--pattern", f"{tile_id}_pre_disaster.png",
        "--max-images", "1",
        "--prompt", "building",
        "--min-size", "30",
        "--output-style", "notebook",
        "--batch-size", "1",
        "--device", STAGE1_DEVICE,
        "--backend", "transformers",
        "--tile-size", "512",
        "--overlap", "64",
    ],
    "Stage 1: Building Instance Extraction",
)

label_dir = stage1_out / "labels"

### Stage 1 Reporting

Verify Stage1 completion artifacts and count instance-level prediction JSON outputs.

In [ ]:
# --- 3b) Stage 1: report ---
stage1_summary = stage1_out / "run_summary.json"
print("[stage1] run_summary_exists=", stage1_summary.exists())
print("[stage1] label_json_count=", len(list(label_dir.glob("*_prediction.json"))) )

### Shared Subimage Generation

Create standardized per-instance crops and masks used by both Stage2a and Stage2b.

In [ ]:
# --- 4a) Shared subimages: execute ---
run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/generate_shared_instance_subimages.py",
        "--stage1_labels_dir", label_dir,
        "--pre_images_dir", pair_dir,
        "--post_images_dir", pair_dir,
        "--out_root", shared_out,
        "--crop_size", "256",
        "--ring_radius_px", "48",
        "--strict_images",
        "--num_workers", "4",
        "--chunk_size", "100",
        "--log_every", "50",
    ],
    "Shared Subimage Generation",
)

shared_csv = shared_out / "shared_instance_samples.csv"

### Shared Subimages Reporting

Check generated shared-instance table size and confidence statistics.

In [ ]:
# --- 4b) Shared subimages: report ---
df_shared = pd.read_csv(shared_csv)
print("[shared] rows=", len(df_shared))
if "sam3_confidence" in df_shared.columns:
    x = pd.to_numeric(df_shared["sam3_confidence"], errors="coerce").dropna()
    if len(x) > 0:
        print("[shared] sam3_confidence median=", round(float(np.median(x)), 4))

### Stage 2a Inference

Build Stage2a input CSV from shared artifacts and run type/population inference.

In [ ]:
# --- 5a) Stage 2a: execute ---
run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/build_stage2a_infer_csv.py",
        "--shared_csv", shared_csv,
        "--out_csv", stage2a_input_csv,
        "--crop_col", "pre_crop",
        "--mask_col", "mask_M",
    ],
    "Stage 2a: Build Inference CSV",
)

run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/infer_stage2a.py",
        "--input_csv", stage2a_input_csv,
        "--ckpt", PACKAGE_ROOT / "models/stage2a/stage2a_best_model.pt",
        "--out_csv", stage2a_out_csv,
        "--batch_size", "4",
        "--num_workers", "4",
        "--device", STAGE_DEVICE,
        "--print_examples", "5",
    ],
    "Stage 2a: Population/Type Inference",
)

df_s2a = pd.read_csv(stage2a_out_csv)

### Stage 2a Reporting

Summarize Stage2a prediction distribution and population quantiles.

In [ ]:
# --- 5b) Stage 2a: report ---
print("[stage2a] rows=", len(df_s2a))
if "pred_type_class" in df_s2a.columns:
    print("[stage2a] type_counts=", df_s2a["pred_type_class"].value_counts().to_dict())
if "pred_population" in df_s2a.columns:
    p = pd.to_numeric(df_s2a["pred_population"], errors="coerce").dropna()
    if len(p) > 0:
        q = np.percentile(p, [50, 90, 99])
        print(f"[stage2a] population p50/p90/p99 = {q[0]:.2f}/{q[1]:.2f}/{q[2]:.2f}")

### Stage 2b Inference

Run calibrated ensemble damage inference and collect per-instance uncertainty outputs.

In [ ]:
# --- 6a) Stage 2b: execute ---
run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/infer_stage2_ensemble.py",
        "--csv", shared_csv,
        "--ckpts", "models/stage2b/inference0.7273.pt,models/stage2b/inference0.7066_seed9999.pt,models/stage2b/inference0.7034_seed7777.pt",
        "--configs", "configs/stage2b/run019_seed2025_train_config.json,configs/stage2b/seed9999_train_config.json,configs/stage2b/seed7777_train_config.json",
        "--weights", "4,3,2",
        "--calibration_dirs", "calibration/calibration_run019_r48,calibration/calibration_seed9999_r48,calibration/calibration_seed7777_r48",
        "--calibration_method", "temperature",
        "--out_jsonl", stage2b_out_jsonl,
        "--batch_size", "2",
        "--num_workers", "4",
        "--device", STAGE_DEVICE,
        "--print_examples", "5",
        "--log_every_steps", "20",
    ],
    "Stage 2b: Damage/Uncertainty Inference",
)

rows_s2b = []
with stage2b_out_jsonl.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows_s2b.append(json.loads(line))

### Stage 2b Reporting

Summarize Stage2b predictions with class counts and confidence diagnostics.

In [ ]:
# --- 6b) Stage 2b: report ---
print("[stage2b] rows_jsonl=", len(rows_s2b))
if rows_s2b:
    y_counts = pd.Series([str(r.get("y_pred_ensemble", "")) for r in rows_s2b]).value_counts().to_dict()
    print("[stage2b] damage_pred_counts=", y_counts)
    pmax_vals = [float(r.get("pmax", np.nan)) for r in rows_s2b]
    pmax_vals = [v for v in pmax_vals if np.isfinite(v)]
    if pmax_vals:
        print("[stage2b] pmax mean=", round(float(np.mean(pmax_vals)), 4))

### Synthesis and Visualization

Run instance-level presentation synthesis and generate overlay visualization artifacts.

In [ ]:
# --- 7a) Synthesis + visualization: execute ---
run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/present_instance_results.py",
        "--shared_csv", shared_csv,
        "--stage2a_csv", stage2a_out_csv,
        "--stage2b_jsonl", stage2b_out_jsonl,
        "--out_csv", presented_csv,
        "--out_top_uncertain_csv", uncertain_csv,
        "--top_k_uncertain", "30",
        "--print_top_n", "15",
    ],
    "Synthesis: Instance-Level Presentation",
)

run_step(
    [
        PYTHON_BIN,
        PACKAGE_ROOT / "scripts/visualize_stage2_overlays.py",
        "--pred_jsonl", stage2b_out_jsonl,
        "--csv", shared_csv,
        "--stage2a_csv", stage2a_out_csv,
        "--out_dir", vis_dir,
        "--max_outputs", "100",
        "--fill_opacity", "0.5",
    ],
    "Visualization: Instance Overlays",
)

print("\n[done] run_dir:", RUN_DIR)
print("[done] stage1_out:", stage1_out)
print("[done] shared_csv:", shared_csv)
print("[done] stage2a_out:", stage2a_out_csv)
print("[done] stage2b_out:", stage2b_out_jsonl)
print("[done] presented_csv:", presented_csv)
print("[done] top_uncertain_csv:", uncertain_csv)
print("[done] vis_dir:", vis_dir)

### Load Presented Tables

Load synthesized output CSVs into DataFrames for reporting, plotting, and visual inspection.

In [ ]:
# --- 7b) Synthesis: report/load ---
df = pd.read_csv(presented_csv)
df_unc = pd.read_csv(uncertain_csv)
print("rows presented:", len(df))
print("rows uncertain:", len(df_unc))

### Final Synthesis Summary

Print compact joined statistics across Stage1, Stage2a, and Stage2b outputs.

In [ ]:
# --- 8) Final synthesis summary ---
print("[summary] all_rows=", len(df))
if "has_stage2a" in df.columns and "has_stage2b" in df.columns:
    joined = int((df["has_stage2a"].astype(bool) & df["has_stage2b"].astype(bool)).sum())
    print("[summary] all_three_joined=", joined)

if "stage2a_pred_type_class" in df.columns:
    print("[summary] counts_by_stage2a_type=", df["stage2a_pred_type_class"].fillna("").value_counts().to_dict())
if "stage2b_pred_damage_class" in df.columns:
    d = df["stage2b_pred_damage_class"].astype(str)
    print("[summary] counts_by_stage2b_damage_class_valid_only=", d[d != ""].value_counts().to_dict())

### Distribution Diagnostics

Plot key output distributions for damage classes, Stage2b confidence, and Stage2a population signals.

In [ ]:
# --- 9) Distributional diagnostics ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if "stage2b_pred_damage_class" in df.columns:
    damage = df["stage2b_pred_damage_class"].astype(str)
    damage = damage[damage != ""]
    damage.value_counts().sort_index().plot(kind="bar", ax=axes[0], title="Damage Class Counts")
    axes[0].set_xlabel("damage class")

if "stage2b_pmax" in df.columns:
    pd.to_numeric(df["stage2b_pmax"], errors="coerce").dropna().plot(kind="hist", bins=20, ax=axes[1], title="Stage2b Confidence (pmax)")
    axes[1].set_xlabel("pmax")

if "stage2a_pred_population" in df.columns:
    pd.to_numeric(df["stage2a_pred_population"], errors="coerce").dropna().plot(kind="hist", bins=20, ax=axes[2], title="Stage2a Predicted Population")
    axes[2].set_xlabel("population")

plt.tight_layout()
plt.show()

### Uncertainty Preview

Display the top uncertain instances and preview generated overlay images for qualitative review.

In [ ]:
# --- 10) Uncertainty table + overlay preview ---
preview_cols = [
    "instance_id",
    "tile_id",
    "stage2b_pred_damage_class",
    "stage2b_pmax",
    "stage2b_entropy",
    "stage2b_var_expected_severity_weighted",
    "stage1_sam3_confidence",
    "stage2a_pred_population",
    "stage2a_pred_type_class",
    "stage2a_pred_type_conf",
]
cols = [c for c in preview_cols if c in df_unc.columns]
print("Top uncertain instances (first 20):")
display(df_unc[cols].head(20))

all_png = sorted(vis_dir.glob("*.png"))
assert len(all_png) > 0, f"No overlay images found in {vis_dir}"
img_paths = all_png[:6]
print("overlay files found:", len(all_png))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
for ax, p in zip(axes, img_paths):
    im = Image.open(p)
    ax.imshow(im)
    ax.set_title(p.name[:56], fontsize=8)
    ax.axis("off")
for ax in axes[len(img_paths):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8) Conclusion Template (for reports)

This case study demonstrates the full instance-impact pathway:

1. **Input**: paired pre/post imagery for a selected tile.
2. **Processing**: instance extraction and stage-wise inference executed end-to-end in this notebook.
3. **Outputs**: instance-level damage, exposure proxies, and uncertainty diagnostics.
4. **Interpretation**: uncertainty-ranked instances and overlay visual evidence support downstream triage.

For comparative studies, repeat with multiple `RUN_ID`s and aggregate the same summary statistics across tiles/events.